In [ ]:
import json
import logging
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from datetime import datetime
import plotly.graph_objects as go

# Import the refactored functions
from src.las_file_handler import process_las_file

In [ ]:
# --- CONFIGURATION ---
# Per the style guide, paths are relative to the project root.
# Notebooks are in notebooks/, so we go up one level.
PROJECT_ROOT = Path.cwd().parent 
SOURCE_DIR = PROJECT_ROOT / 'data' / 'raw'
DEST_DIR = PROJECT_ROOT / 'data' / 'processed'
LOG_DIR = PROJECT_ROOT / 'logs'

# Create directories if they don't exist
SOURCE_DIR.mkdir(exist_ok=True)
DEST_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# --- LOGGING SETUP ---
log_file = LOG_DIR / 'copy_log.log'
logging.basicConfig(filename=log_file, level=logging.INFO, format='%(asctime)s - %(message)s')

print(f"Source Directory: {SOURCE_DIR}")
print(f"Destination Directory: {DEST_DIR}")
print(f"Log Directory: {LOG_DIR}")
print("Configuration and logging initialized.")

Source Directory: /Users/davidthul/Desktop/lasSrcTest
Destination Directory: /Users/davidthul/Desktop/lasDestTest
Log Directory: /Users/davidthul/Desktop/trackingLogs
Configuration and logging initialized.


### Reference Data and Mappings
This cell contains the reference dictionaries used for mapping API codes and county names to basins. Mappings is used for parsing location data from filenames, while is a fallback for parsing from file content.

In [ ]:
# The logic for loading the API map is now part of the main processing cell.
# This cell is no longer needed as the logic is consolidated.
# We will load the JSON file just before we start processing the files.
pass

Loaded basin lookup data from: basic_basin_lookup.json
Version: 2.1, Coverage: Includes portions of DJ and Powder River


### Helper Functions for Location Parsing
These functions are the core of the location identification logic. They either parse the filename for an API number or look inside the LAS file for county information.

In [ ]:
# --- DATA LOADING ---
basin_lookup_path = Path('basic_basin_lookup.json')
with open(basin_lookup_path, 'r') as f:
    API_MAP = json.load(f)

# --- FILE PROCESSING ---
all_files = [p for p in SOURCE_DIR.iterdir() if p.is_file() and p.suffix.lower() == '.las']
print(f"📂 Found {len(all_files)} files. Starting processing...")

# Use a list comprehension with tqdm for progress tracking
tracking_data = [process_las_file(log_file, API_MAP, DEST_DIR) for log_file in tqdm(all_files, desc="Processing LAS Files")]

# --- SAVE RESULTS ---
if tracking_data:
    df = pd.DataFrame(tracking_data)
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = LOG_DIR / f"copy_log_{timestamp_str}.parquet"
    
    # Use Polars for saving to Parquet as per style guide for larger data
    try:
        import polars as pl
        pl.from_pandas(df).write_parquet(save_path)
        print(f"✅ SUCCESS! Log saved to: {save_path} using Polars.")
    except ImportError:
        df.to_parquet(save_path)
        print(f"✅ SUCCESS! Log saved to: {save_path} using Pandas.")

    print("\n--- Sample of Processed Data ---")
    print(df[['method', 'county', 'basin', 'status']].head())
else:
    print("❌ No data collected.")

📂 Found 201 files. Processing batch of 201...


Processing:   0%|          | 0/201 [00:00<?, ?it/s]

✅ SUCCESS! Log saved to: /Users/davidthul/Desktop/trackingLogs/copy_log_20260112_143200.parquet
       method    county         basin
0  API-49-009  Converse  Powder River
1  API-49-009  Converse  Powder River
2  API-49-035  Sublette   Green River
3  API-49-035  Sublette   Green River
4  API-49-009  Converse  Powder River


### Analysis and Visualization
This final section loads all the tracking data from the Parquet files, processes it, and generates a Sankey diagram to visualize how files were categorized.

In [ ]:
import polars as pl
import plotly.graph_objects as go

# 1. Load Data using Polars
all_tracking_files = list(LOG_DIR.glob('*.parquet'))

if not all_tracking_files:
    print("No tracking files found.")
else:
    # Load all parquet files into a single Polars DataFrame
    df = pl.read_parquet(sorted(all_tracking_files))

    # Fill nulls
    df = df.with_columns(
        pl.col('method').fill_null('Unknown Method'),
        pl.col('county').fill_null('Unknown County'),
        pl.col('basin').fill_null('Uncategorized')
    )

    # 2. Prepare Data for Sankey using Polars
    flow_1 = df.group_by(['method', 'county']).agg(pl.count().alias('count')).rename({'method': 'source', 'county': 'target', 'count': 'value'})
    flow_2 = df.group_by(['county', 'basin']).agg(pl.count().alias('count')).rename({'county': 'source', 'basin': 'target', 'count': 'value'})

    # 3. Create Node List
    unique_methods = [f"{m} (Method)" for m in flow_1['source'].unique().to_list()]
    
    # Combine unique counties from both flows
    county_set = set(flow_1['target'].unique().to_list() + flow_2['source'].unique().to_list())
    unique_counties = [f"{c} (County)" for c in county_set]
    
    unique_basins = [f"{b} (Basin)" for b in flow_2['target'].unique().to_list()]
    
    all_nodes = unique_methods + unique_counties + unique_basins
    node_map = {name: i for i, name in enumerate(all_nodes)}

    def get_node_idx(raw_name, category_suffix):
        return node_map.get(f"{raw_name} ({category_suffix})")

    # 4. Map Sources and Targets
    source_1 = flow_1['source'].apply(lambda x: get_node_idx(x, 'Method')).to_list()
    target_1 = flow_1['target'].apply(lambda x: get_node_idx(x, 'County')).to_list()
    value_1 = flow_1['value'].to_list()
    
    source_2 = flow_2['source'].apply(lambda x: get_node_idx(x, 'County')).to_list()
    target_2 = flow_2['target'].apply(lambda x: get_node_idx(x, 'Basin')).to_list()
    value_2 = flow_2['value'].to_list()

    final_sources = source_1 + source_2
    final_targets = target_1 + target_2
    final_values = value_1 + value_2

    # 5. Clean Labels for Display
    display_labels = [label.rsplit(' (', 1)[0] for label in all_nodes]

    # 6. Plot
    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5), label=display_labels, color="blue"),
        link=dict(source=final_sources, target=final_targets, value=final_values, color='rgba(31, 119, 180, 0.4)')
    )])

    fig.update_layout(title_text="Data Flow: Method ➔ County ➔ Basin", font_size=12, height=600)
    fig.show()